# Conflict Early Warning Nigeria — Phase 4
## Predictive Modelling, Expanding-Window Backtesting & Calibration

**Goal:** Build, evaluate, and statistically validate predictive models for conflict risk across Nigeria's 36 states and FCT, with a pilot focus on the Northwest (Zamfara, Katsina, Sokoto, Kaduna).

**Target Variable:**  
> $P(\text{fatal\_event}_{T+1} \ge 1 \mid \text{State } s)$ — probability of at least one fatal conflict event in month $T+1$.

**Model Ladder:**
1. **Baseline / Climatology:** Constant empirical base rate prevalence
2. **M0 — Naive Persistence:** Last observed month = forecast ($y_T = \hat{p}_{T+1}$)
3. **M1 — Negative-Binomial GLM:** Count regression on fatalities $\to$ zero-event probability exceedance
4. **M2 — LightGBM:** Gradient-boosted decision trees capturing non-linear interactions & spatial spillovers

**Evaluation Protocol:**
- Strict expanding-window backtest (83 monthly folds, 2018–2024)
- Zero future leakage: model at fold $T$ trains strictly on data prior to $T$
- Statistical validation via Diebold-Mariano test (1995 with HLN 1997 correction)
- Calibration assessment: Reliability diagrams and Expected Calibration Error (ECE)


### 1 · Imports and Setup

In [ ]:
import pathlib, sys, json, warnings
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from sklearn.metrics import roc_curve, auc, brier_score_loss
from sklearn.calibration import calibration_curve

warnings.filterwarnings('ignore')
ROOT = pathlib.Path('..').resolve()
sys.path.insert(0, str(ROOT))

from src.models.baseline import NaiveM0, ClimatologicalBaseline
from src.models.glm import NegativeBinomialM1, DEFAULT_GLM_FEATURES
from src.models.gbm import LightGBMM2, DEFAULT_GBM_FEATURES
from src.models.backtest import ExpandingWindowBacktester
from src.models.evaluation import (
    brier_score, brier_skill_score, expected_calibration_error,
    brier_decomposition, diebold_mariano_test
)

FEAT_PATH = ROOT / 'data' / 'features' / 'feature_matrix.parquet'
print('Setup complete. Ready to run Phase 4 models.')


### 2 · Load Feature Matrix

In [ ]:
df = pd.read_parquet(FEAT_PATH)
df = df.sort_values(['admin1', 'month']).reset_index(drop=True)
print(f'Feature matrix loaded: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'Date range: {df["month"].min()} to {df["month"].max()}')
print(f'States: {df["admin1"].nunique()}')


### 3 · Execute Expanding-Window Backtesting
We evaluate 83 expanding window folds starting after a 36-month burn-in period.

In [ ]:
backtester = ExpandingWindowBacktester(
    min_train_months=36,
    burn_in_months=12,
    glm_features=DEFAULT_GLM_FEATURES,
    gbm_features=DEFAULT_GBM_FEATURES,
)
pred_df = backtester.run(df)
metrics = backtester.metrics_
print(f'Backtest completed across {metrics["folds_evaluated"]} folds ({len(pred_df)} test predictions).')


### 4 · Benchmark Comparison and Statistical Tests

In [ ]:
m = metrics['models']
summary_table = pd.DataFrame([
    {'Model': 'Climatology', 'Brier': m['climatology']['brier_score'], 'BSS': 0.0, 'AUC': 0.500, 'ECE': 'N/A'},
    {'Model': 'M0 Naive', 'Brier': m['M0_naive']['brier_score'], 'BSS': m['M0_naive']['brier_skill_score'], 'AUC': m['M0_naive']['auc_roc'], 'ECE': m['M0_naive']['ece']},
    {'Model': 'M1 NegBin GLM', 'Brier': m['M1_negbin_glm']['brier_score'], 'BSS': m['M1_negbin_glm']['brier_skill_score'], 'AUC': m['M1_negbin_glm']['auc_roc'], 'ECE': m['M1_negbin_glm']['ece']},
    {'Model': 'M2 LightGBM', 'Brier': m['M2_lightgbm']['brier_score'], 'BSS': m['M2_lightgbm']['brier_skill_score'], 'AUC': m['M2_lightgbm']['auc_roc'], 'ECE': m['M2_lightgbm']['ece']},
])
display(summary_table)

dm_m1 = m['M1_negbin_glm']['dm_test_vs_m0']
dm_m2 = m['M2_lightgbm']['dm_test_vs_m0']
print(f"Diebold-Mariano M1 vs M0: DM={dm_m1['dm_stat']:+.2f}, p={dm_m1['p_value']:.4e} (Significant: {dm_m1['significant']})")
print(f"Diebold-Mariano M2 vs M0: DM={dm_m2['dm_stat']:+.2f}, p={dm_m2['p_value']:.4e} (Significant: {dm_m2['significant']})")


### 5 · Reliability and Calibration Diagnostics

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ax.plot([0, 1], [0, 1], 'k--', alpha=0.7, label='Perfect Calibration')
for mod_name, col, mark in [('p_m0', '#e74c3c', 's'), ('p_m1', '#2980b9', 'o'), ('p_m2', '#27ae60', '^')]:
    f_pos, m_pred = calibration_curve(pred_df['actual'], pred_df[mod_name], n_bins=10, strategy='uniform')
    ax.plot(m_pred, f_pos, marker=mark, label=mod_name, color=col)
ax.set_xlabel('Predicted Probability')
ax.set_ylabel('Observed Event Rate')
ax.set_title('Reliability Diagram (Calibration Curves)')
ax.legend()
ax.grid(True, alpha=0.4)
plt.show()


### 6 · Subgroup Analysis: Northwest Pilot States

In [ ]:
nw = metrics['subgroups']['northwest_pilot']
print(f"Northwest Pilot States (Zamfara, Katsina, Sokoto, Kaduna):")
print(f"  Prevalence : {nw['prevalence']:.1%}")
print(f"  M0 Brier   : {nw['brier_m0']:.4f}")
print(f"  M1 Brier   : {nw['brier_m1']:.4f}")
print(f"  M2 Brier   : {nw['brier_m2']:.4f}")
